In [ ]:
import pickle
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import tqdm

from tklearn.kb import KnowledgeBase
from tklearn.kb.wiktionary.models import Word, parse_jsonl

In [ ]:
kb = KnowledgeBase("wiktionary")

In [ ]:
words: list[Word] = kb.load_words()

In [ ]:
senses = []

predicate_type_count = {
    "synonym": 0,
    "antonym": 0,
    "hypernym": 0,
    "hyponym": 0,
}
triples = set()
sense_ids = {}
definition_words = defaultdict(set)
form2words = defaultdict(set)

for word in tqdm.tqdm(words):
    form2words[word.word].add((word.word, None))
    for form in word.forms or []:
        word_form: str = form.form
        form2words[word_form].add((word.word, None))
    for sense in word.senses or []:
        # Add gloss-based sense ID
        sense_id = len(sense_ids)
        for definition in sense.glosses or []:
            if definition in sense_ids:
                sense_id = sense_ids[definition]
                break
        for definition in sense.glosses or []:
            sense_ids[definition] = sense_id
            definition_words[definition].add(word.word)
        # Add form-of relations
        for form_of in sense.form_of or []:
            form2words[word.word].add((form_of.word, sense_id))
        # Add other relations
        for predicate in predicate_type_count.keys():
            objects = getattr(sense, predicate + "s") or []
            for obj in objects:
                triple = (
                    (word.word, sense_id),
                    predicate,
                    (obj.word, obj.sense),
                )
                triples.add(triple)
            predicate_type_count[predicate] += len(objects)

predicate_type_count
# original: Counter({'synonym': 151086, 'antonym': 16029, 'hyponym': 842})
# en-sm: {'synonym': 423210, 'antonym': 16494, 'hypernym': 12118, 'hyponym': 3477}

In [ ]:
df = pd.DataFrame(
    [
        (x, z[0], int(z[1]) if z[1] else -1)
        for x, y in form2words.items()
        for z in y
    ],
    columns=["form", "word", "sense_id"],
)
df.dropna(inplace=True)
df = df.groupby(["form", "word"]).agg({"sense_id": set}).reset_index()

In [ ]:
df.loc[df["form"] == "dog"]

In [ ]:
df.loc[df["word"] == "dog"]

In [ ]:
definitions_df = pd.DataFrame(
    [
        (definition, sense, definition_words[definition])
        for definition, sense in sense_ids.items()
    ],
    columns=["definition", "sense_id", "words"],
)
definitions_df.sort_values("definition", ascending=False, inplace=True)
definitions_df[definitions_df["sense_id"].duplicated(keep=False)]

In [ ]:
triples_df = pd.DataFrame(triples, columns=["subject", "predicate", "object"])
# subject -> subject.word, subject.sense
# object -> object.word, object.sense
triples_df = triples_df.assign(
    # subject_word, subject_sense, predicate, object_word, object_sense
    subject_word=triples_df["subject"].apply(lambda x: x[0]),
    subject_sense=triples_df["subject"].apply(lambda x: x[1]),
    object_word=triples_df["object"].apply(lambda x: x[0]),
    object_sense=triples_df["object"].apply(lambda x: x[1]),
).sort_values(["subject_word", "predicate", "object_word"])
triples_df.drop(columns=["subject", "object"], inplace=True)
triples_df.head()
assert not triples_df.duplicated().any(), "Duplicates in triples_df"

In [ ]:
triples_df[
    triples_df.duplicated(
        subset=["subject_word", "object_word", "predicate"], keep=False
    )
].head(20)

In [ ]:
sense_id2definitions = defaultdict(set)
for _, row in definitions_df.iterrows():
    sense_id2definitions[row["sense_id"]].add(row["definition"])

In [ ]:
sense_id2definitions[1239826]

In [ ]:
sense_id2definitions[1239825]

In [ ]:
# save all the definitions in a cache file
cache_dir = Path("../cache")

list(cache_dir.iterdir())

In [ ]:
definitions_df.reset_index(drop=True).to_parquet(
    cache_dir / "wiktionary_definitions.parquet"
)

In [ ]:
definitions_df